# RoBERTa Fine-tuning with W&B Hyperparameter Sweep


| Mode | How to run |
|------|------------|
| Debug (CPU, 200 samples) | Set `DEBUG = True` in the Config cell |
| Single run | Set `MODE = "single"` |
| W&B Sweep | Set `MODE = "sweep"` |

Currently training on **RTX 4090 24GB**

## 1. Imports

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [ ]:
import json
from pathlib import Path
from typing import Optional

import numpy as np
import wandb
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

## 2. Config

**Edit this cell** to switch between modes and adjust hyperparameters.

In [ ]:
DEBUG = False     # True → CPU smoke-test with 200 samples
# MODE = "single"  # "single" | "sweep"
MODE = "sweep"    # "single" | "sweep"
SWEEP_COUNT = 20  # number of sweep trials (only used when MODE="sweep")

BASE_MODEL = "roberta-base"
MAX_LENGTH = 128
WANDB_PROJECT = "RoBERTa-Finetuning"

NOTEBOOK_DIR = Path(".").resolve()
DATA_ROOT = NOTEBOOK_DIR.parent / "data"
SAVE_DIR = NOTEBOOK_DIR / "roberta-finetuned" / "best_model"

DEFAULT_HP = dict(
    learning_rate = 2e-5,
    # per_device_train_batch_size = 32, # Use smaller batch sizes if GPU memory is a concern; adjust as needed
    per_device_train_batch_size = 64, # Use larger batch sizes if GPU memory allows (24GB or larger); adjust as needed
    num_train_epochs = 3,
    weight_decay = 0.01,
    warmup_ratio = 0.06,
)

print(f"Mode: {'DEBUG ' if DEBUG else ''}{MODE.upper()}")
print(f"Data root : {DATA_ROOT}")
print(f"Model save: {SAVE_DIR}")

Mode: SWEEP
Data root : /root/autodl-tmp/programs/comp6713-sentiment-analysis/data
Model save: /root/autodl-tmp/programs/comp6713-sentiment-analysis/models/roberta-finetuned/best_model


## 3. W&B Sweep Configuration

Uses **Bayesian optimisation** to search for the best hyperparameter combination.

In [ ]:
SWEEP_CONFIG = {
    "method": "bayes",
    "metric": {"name": "eval/f1", "goal": "maximize"},
    "parameters": {
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-5,
            "max": 5e-5,
        },
        # "per_device_train_batch_size": {"values": [16, 32]},  # Use smaller batch sizes if GPU memory is a concern; adjust as needed
        "per_device_train_batch_size": {"values": [32, 64, 128]},  # Use larger batch sizes if GPU memory allows (24GB or larger); adjust as needed
        "num_train_epochs": {"values": [2, 3, 4]},
        "weight_decay": {"values": [0.0, 0.01, 0.1]},
        "warmup_ratio": {
            "distribution": "uniform",
            "min": 0.0,
            "max": 0.2,
        },
    },
}

print("Sweep method:", SWEEP_CONFIG["method"])
print("Optimising: ", SWEEP_CONFIG["metric"])
for k, v in SWEEP_CONFIG["parameters"].items():
    print(f"  {k}: {v}")

Sweep method: bayes
Optimising:  {'name': 'eval/f1', 'goal': 'maximize'}
  learning_rate: {'distribution': 'log_uniform_values', 'min': 1e-05, 'max': 5e-05}
  per_device_train_batch_size: {'values': [32, 64, 128]}
  num_train_epochs: {'values': [2, 3, 4]}
  weight_decay: {'values': [0.0, 0.01, 0.1]}
  warmup_ratio: {'distribution': 'uniform', 'min': 0.0, 'max': 0.2}


## 4. Data Loading

In [ ]:
with open(DATA_ROOT / "train.json", encoding="utf-8") as f:
    train_records = json.load(f)

with open(DATA_ROOT / "val.json", encoding="utf-8") as f:
    val_records = json.load(f)

if DEBUG:
    train_records = train_records[:200]
    val_records = val_records[:50]

print(f"Train: {len(train_records)} samples | Val: {len(val_records)} samples")
print("Sample record:", train_records[0])

train_ds_raw = Dataset.from_list(train_records)
val_ds_raw = Dataset.from_list(val_records)

Train: 20484 samples | Val: 5121 samples
Sample record: {'text': "It looks like it's from the Molecule Effect on Santa Fe. Great quirky spot!", 'label': 1}


## 5. Tokenisation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, mirror="https://hf-mirror.com")


def apply_tokenizer(dataset: Dataset) -> Dataset:
    return dataset.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        ),
        batched=True,
        remove_columns=["text"],
    )


train_ds = apply_tokenizer(train_ds_raw)
val_ds = apply_tokenizer(val_ds_raw)

print("Train features:", train_ds.features)
print("Val   features:", val_ds.features)

Map:   0%|          | 0/20484 [00:00<?, ? examples/s]

Map:   0%|          | 0/5121 [00:00<?, ? examples/s]

Train features: {'label': Value(dtype='int64', id=None), 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None), 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}
Val   features: {'label': Value(dtype='int64', id=None), 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None), 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}


## 6. Metrics

In [ ]:
def compute_metrics(eval_pred) -> dict:
    preds  = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary"),
    }

## 7. Core Training Function

In [ ]:
def run_training(
    *,
    learning_rate: float = 2e-5,
    per_device_train_batch_size: int = 32,
    num_train_epochs: int = 3,
    weight_decay: float = 0.01,
    warmup_ratio: float = 0.06,
    debug: bool = False,
    run_name: Optional[str] = None,
    save_best: bool = True,
) -> None:

    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
    args = TrainingArguments(
        output_dir=str(SAVE_DIR.parent / "checkpoints"),
        num_train_epochs=1 if debug else num_train_epochs,
        per_device_train_batch_size=16 if debug else per_device_train_batch_size,
        # per_device_eval_batch_size=64, # Use smaller eval batch size if GPU memory is a concern; adjust as needed
        per_device_eval_batch_size=128, # Use larger eval batch size if GPU memory allows (24GB or larger); adjust as needed
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        eval_strategy="epoch",
        save_strategy="epoch",  # Changed to "epoch" to match eval_strategy
        save_total_limit=1,  # Keep only the best checkpoint to save space
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="wandb",
        logging_steps=10 if debug else 50,
        no_cuda=debug,
        run_name=run_name,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()

    if save_best:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(SAVE_DIR))
        tokenizer.save_pretrained(str(SAVE_DIR))
        print(f"\nModel saved → {SAVE_DIR}")

## 8. Run Training

Executes **single run** or **sweep** based on the `MODE` set in the Config cell.

In [9]:
def sweep_run() -> None:
    """Called once per sweep trial by the wandb agent."""
    with wandb.init() as run:
        cfg = run.config
        run_training(
            learning_rate=cfg.learning_rate,
            per_device_train_batch_size=cfg.per_device_train_batch_size,
            num_train_epochs=cfg.num_train_epochs,
            weight_decay=cfg.weight_decay,
            warmup_ratio=cfg.warmup_ratio,
            debug=False,
            run_name=run.name,
            save_best=True,
        )


if MODE == "sweep":
    sweep_id = wandb.sweep(SWEEP_CONFIG, project=WANDB_PROJECT)
    print(f"Sweep ID: {sweep_id}  (count={SWEEP_COUNT})")
    print("View at: https://wandb.ai/home → project RoBERTa-Finetuning")
    wandb.agent(sweep_id, function=sweep_run, count=SWEEP_COUNT)

else:  # single
    run_name = "roberta-debug" if DEBUG else "roberta-finetune-v1"
    wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={"model": BASE_MODEL, "max_length": MAX_LENGTH, "debug": DEBUG, **DEFAULT_HP},
    )
    run_training(debug=DEBUG, run_name=run_name, save_best=not DEBUG, **DEFAULT_HP)
    wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Create sweep with ID: jaq6295f
Sweep URL: https://wandb.ai/zkgao223-unsw/RoBERTa-Finetuning/sweeps/jaq6295f
Sweep ID: jaq6295f  (count=20)
View at: https://wandb.ai/home → project RoBERTa-Finetuning


wandb: Agent Starting Run: p67lig39 with config:
wandb: 	learning_rate: 3.530305312987959e-05
wandb: 	num_train_epochs: 2
wandb: 	per_device_train_batch_size: 128
wandb: 	warmup_ratio: 0.0010255231873944391
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: zkgao223 (zkgao223-unsw) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[RANK 0] Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING Config item 'per_device_train_batch_size' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'weight_decay' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'num_train_epochs' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'warmup_ratio' was locked by 'sweep' (ignored update).
wandb: WAR

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.231500,0.215386,0.914275,0.932762
2,0.178500,0.211512,0.918375,0.936953


...

wandb: Agent Starting Run: ka7dad8k with config:
wandb: 	learning_rate: 4.083392327605057e-05
wandb: 	num_train_epochs: 4
wandb: 	per_device_train_batch_size: 64
wandb: 	warmup_ratio: 0.07932351167057916
wandb: 	weight_decay: 0.1
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[RANK 0] Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING Config item 'per_device_train_batch_size' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'weight_decay' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'num_train_epochs' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'warmup_ratio' was locked by 'sweep' (ignored update).


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.246700,0.237242,0.910174,0.931892
2,0.188400,0.232322,0.913884,0.934560
3,0.114400,0.255152,0.914860,0.934377
4,0.081500,0.294738,0.916227,0.935846



Model saved → /root/autodl-tmp/programs/comp6713-sentiment-analysis/models/roberta-finetuned/best_model


eval/accuracy,▁▅▆█
eval/f1,▁▆▅█
eval/loss,▂▁▄█
eval/runtime,▁▆▅█
eval/samples_per_second,█▃▄▁
eval/steps_per_second,█▃▄▁
train/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇████
train/global_step,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇████
train/grad_norm,▃▂▂▂▃▁▂▁▂▃▄▃▂▄▃▇▂▁▄█▂▅▄▆▂
train/learning_rate,▄███▇▇▇▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁
+1,...


## 9. Inference

Load the saved model and run predictions. Run this cell independently after training.

In [ ]:
import torch

test_tokenizer = AutoTokenizer.from_pretrained(str(SAVE_DIR))
test_model = AutoModelForSequenceClassification.from_pretrained(str(SAVE_DIR))
test_model.eval()

def predict(text: str) -> dict:
    inputs = test_tokenizer(
        text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )
    with torch.no_grad():
        logits = test_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    pred = torch.argmax(probs).item()
    return {
        "label": pred,
        "confidence": round(probs[pred].item(), 4),
        "label_text": "positive" if pred == 1 else "negative",
    }

# Simple test
samples = [
    "The food was absolutely amazing, best meal I've ever had!",
    "Terrible service, waited an hour and the food was cold.",
]
for s in samples:
    result = predict(s)
    print(f"[{result['label_text'].upper():8s} {result['confidence']:.2%}] {s}")

[POSITIVE 99.96%] The food was absolutely amazing, best meal I've ever had!
[NEGATIVE 99.75%] Terrible service, waited an hour and the food was cold.
